### Preprocessing

Import Necessary Libraries

In [ ]:
import pandas as pd
from collections import Counter
import numpy as np
from tensorflow.keras.preprocessing.sequence import pad_sequences
import matplotlib.pyplot as plt
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight

# File paths
sequence_file = "all_analysis_data.txt"   
label_file = "labels.csv"                

Read the file

In [ ]:
sequences = []
with open(sequence_file, "r") as f:
    for line in f:
        tokens = line.strip().lower().split()
        sequences.append(tokens)

print(f"Total samples: {len(sequences)}")
print(f"Example sequence: {sequences[0][:20]}")

Collapse consecutive duplicates

In [ ]:
def collapse_duplicates(seq):
    if not seq:
        return []
    collapsed = [seq[0]]
    for token in seq[1:]:
        if token != collapsed[-1]:
            collapsed.append(token)
    return collapsed

# Apply to all sequences
sequences = [collapse_duplicates(seq) for seq in sequences]

print(f"Example after collapsing duplicates: {sequences[0][:20]}")


Abstract APIs to behavior categories

In [ ]:
#Flatten and get unique APIs

all_apis = [api for seq in sequences for api in seq]

api_counter = Counter(all_apis)

print(f"Total unique APIs: {len(api_counter)}")
api_counter


In [ ]:
def api_to_behavior(api):
    if api.startswith("ldr"):
        return "LOADER"

    elif api.startswith("reg") or "key" in api:
        return "REGISTRY"

    elif "file" in api or api.startswith("ntqueryattributesfile"):
        return "FILE"

    elif "virtualmemory" in api or "allocate" in api or "free" in api:
        return "MEMORY"

    elif "process" in api:
        return "PROCESS"

    elif "socket" in api or "connect" in api or "send" in api or "recv" in api:
        return "NETWORK"

    elif "crypt" in api or "hash" in api:
        return "CRYPTO"

    elif "string" in api or "loadstring" in api:
        return "STRING"

    else:
        return "OTHER"


In [ ]:
behavior_sequences = []

for seq in sequences:
    behavior_seq = [api_to_behavior(api) for api in seq]
    behavior_sequences.append(behavior_seq)

print(behavior_sequences[0][:30])


In [ ]:
def collapse_duplicates(seq):
    if not seq:
        return []
    collapsed = [seq[0]]
    for item in seq[1:]:
        if item != collapsed[-1]:
            collapsed.append(item)
    return collapsed

behavior_sequences = [
    collapse_duplicates(seq)
    for seq in behavior_sequences
]

print(behavior_sequences[0])


In [ ]:
unique_behaviors = set(
    b for seq in behavior_sequences for b in seq
)

print("Unique behaviors:", unique_behaviors)
print("Count:", len(unique_behaviors))


Vocabulary, Encoding, Sequence Stats, Padding

In [ ]:
behaviors = [
    'LOADER', 'FILE', 'REGISTRY', 'PROCESS',
    'MEMORY', 'NETWORK', 'CRYPTO', 'STRING', 'OTHER'
]


PAD_TOKEN = 0  # For padding
behavior2id = {b: i+1 for i, b in enumerate(behaviors)}  # LOADER=1, FILE=2, ...
id2behavior = {i+1: b for i, b in enumerate(behaviors)} # 1: Loader ....

#Encode behavior sequences to integers
def encode_sequence(seq, mapping):
    return [mapping.get(x, mapping['OTHER']) for x in seq]

encoded_sequences = [encode_sequence(seq, behavior2id) for seq in behavior_sequences]

In [ ]:
#Sequence length statistics

lengths = [len(seq) for seq in encoded_sequences]
print("Min length:", np.min(lengths))
print("Max length:", np.max(lengths))
print("Mean length:", np.mean(lengths))
print("Median length:", np.median(lengths))


plt.hist(lengths, bins=50)
plt.xlabel("Sequence Length")
plt.ylabel("Count")
plt.title("Behavior Sequence Length Distribution")
plt.show()


In [ ]:
#Choose MAX_SEQ_LEN and pad/truncate

MAX_SEQ_LEN = 128  # Adjust based on stats above

X = pad_sequences(
    encoded_sequences,
    maxlen=MAX_SEQ_LEN,
    padding='pre',      # pad at beginning
    truncating='pre',   # truncate from beginning
    value=PAD_TOKEN
)

print("Shape of X:", X.shape)
print("Example padded sequence (first sample):", X[0])


In [ ]:
#save preprocessed dataset
np.save("X_behavior.npy", X)

label encoding

In [ ]:
labels_df = pd.read_csv("labels.csv") 
families = labels_df["Family"].values

In [ ]:
families.value_counts()

In [ ]:
#encode label

label_encoder = LabelEncoder()
y = label_encoder.fit_transform(families)

print(label_encoder.classes_)



In [ ]:
np.save("y.npy", y)

In [ ]:
print("X shape:", X.shape)
print("y shape:", y.shape)


In [ ]:
# First split: train + temp
X = np.load("X.npy")
y = np.load("y.npy")
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y,
    test_size=0.3,
    stratify=y,
    random_state=42
)

# Second split: validation + test
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp,
    test_size=0.5,
    stratify=y_temp,
    random_state=42
)

print("Train:", X_train.shape)
print("Validation:", X_val.shape)
print("Test:", X_test.shape)


In [ ]:
class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_train),
    y=y_train
)

class_weight_dict = dict(zip(np.unique(y_train), class_weights))
print(class_weight_dict)


Build Model